<a href="https://colab.research.google.com/github/catchshashank/optimal-nego/blob/main/stage1/stage1_sst_multifile_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1 SST Embedding Pipeline — Multi-file Colab Notebook

This notebook standardizes and processes **three negotiation CSV files** in one run:

1. `dealerships-nego.csv`
2. `emaad-sales.csv`
3. `heddaya-nego.csv`

It generates the Stage 1 structured output report and the downstream CSV outputs for each dataset separately.

**Design choice:** the notebook saves the full **28-label GoEmotions model output** including `neutral`, while the SST semantic space uses the standard **27 emotion dimensions** by excluding `neutral`. This keeps the neural features intact while preserving the Stage 1 SST-style output format.


In [1]:
# ============================================================
# 0. COLAB SETUP
# ============================================================

!pip install -q transformers torch tqdm scikit-learn scipy

import os
import re
import json
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cross_decomposition import CCA
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore")

print("Setup complete.")
print("CUDA available:", torch.cuda.is_available())


Setup complete.
CUDA available: False


In [2]:
!git clone https://github.com/catchshashank/optimal-nego.git

Cloning into 'optimal-nego'...
remote: Enumerating objects: 179, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 179 (delta 79), reused 48 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (179/179), 1.60 MiB | 4.98 MiB/s, done.
Resolving deltas: 100% (79/79), done.


## 1. Paths and dataset configuration

Upload the three CSV files to:

```text
/content/optimal-nego/data/
```

Expected filenames:

```text
dealerships-nego.csv
emaad-sales.csv
heddaya-nego.csv
```


In [3]:
# ============================================================
# 1. PATHS AND DATASET CONFIG
# ============================================================

BASE_DIR = Path("/content/optimal-nego")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "dealerships_nego": {
        "path": DATA_DIR / "dealerships-nego.csv",
        "schema": "standard",
        "conversation_col": "conversation_id",
        "speaker_col": "speaker_id",
        "text_col": "text",
        "start_col": "start_time",
        "end_col": "end_time",
        "outcome_col": "outcome",
    },
    "emaad_sales": {
        "path": DATA_DIR / "emaad-sales.csv",
        "schema": "standard",
        "conversation_col": "conversation_id",
        "speaker_col": "speaker_id",
        "text_col": "text",
        "start_col": "start_time",
        "end_col": "end_time",
        "outcome_col": "outcome",
    },
    "heddaya_nego": {
        "path": DATA_DIR / "heddaya-nego.csv",
        "schema": "heddaya",
        "conversation_col": "conv_id",
        "speaker_col": "speaker_id",
        "role_col": "role",
        "text_col": "word",
        "turn_col": "turn_id",
        "duration_col": "duration_min",
        "outcome_col": None,
    },
}

# Role mapping used when no explicit role column exists.
# Based on your current negotiation convention:
#   0 = buyer/customer/client
#   1 = seller/dealer/agent
SPEAKER_ID_TO_ROLE = {
    0: "buyer",
    1: "seller",
}

# Keep raw GoEmotions 28 outputs, but build SST from 27 emotion dimensions by excluding neutral.
EXCLUDE_NEUTRAL_FROM_SST = True

# Split-half dimensionality discovery
N_SPLITS = 50
RELIABILITY_THRESHOLD = 0.05
RANDOM_SEED = 42

# GoEmotions inference
MODEL_NAME = "SamLowe/roberta-base-go_emotions"
BATCH_SIZE = 32
MAX_LENGTH = 256

# Progress bar visibility
DISABLE_PROGRESS_BARS = False

print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)


Data directory: /content/optimal-nego/data
Output directory: /content/optimal-nego/outputs


In [4]:
# ============================================================
# 2. STRUCTURE SCAN
# ============================================================

def scan_file_structure(dataset_name, cfg):
    path = cfg["path"]
    if not path.exists():
        return {
            "dataset": dataset_name,
            "exists": False,
            "path": str(path),
            "rows": None,
            "columns": None,
            "column_names": None,
        }

    df_head = pd.read_csv(path, nrows=5)
    row_count = sum(1 for _ in open(path, "r", encoding="utf-8-sig", errors="ignore")) - 1

    return {
        "dataset": dataset_name,
        "exists": True,
        "path": str(path),
        "rows": row_count,
        "columns": df_head.shape[1],
        "column_names": list(df_head.columns),
        "dtypes_preview": {k: str(v) for k, v in df_head.dtypes.items()},
    }

structure_rows = [scan_file_structure(name, cfg) for name, cfg in DATASETS.items()]
structure_df = pd.DataFrame(structure_rows)
structure_df


,dataset,exists,path,rows,columns,column_names,dtypes_preview
0,dealerships_nego,True,/content/optimal-nego/data/dealerships-nego.csv,2496,6,"[conversation_id, speaker_id, start_time, end_...","{'conversation_id': 'object', 'speaker_id': 'i..."
1,emaad_sales,True,/content/optimal-nego/data/emaad-sales.csv,3414,6,"[conversation_id, speaker_id, start_time, end_...","{'conversation_id': 'object', 'speaker_id': 'i..."
2,heddaya_nego,True,/content/optimal-nego/data/heddaya-nego.csv,7565,6,"[conv_id, turn_id, role, word, speaker_id, dur...","{'conv_id': 'int64', 'turn_id': 'int64', 'role..."


In [5]:
# ============================================================
# 3. STANDARDIZATION FUNCTIONS
# ============================================================

BACKCHANNEL_PATTERN = re.compile(
    r"^\s*(uh|um|umm|uhh|hm|hmm|mhm|mm-hmm|yeah|yep|yes|okay|ok|right|sure|alright|got it|thanks|thank you|no problem)[\.!?\s]*$",
    flags=re.IGNORECASE,
)

def is_backchannel(text):
    text = "" if pd.isna(text) else str(text).strip()
    if not text:
        return True
    # Very short standalone acknowledgement-like turns.
    if BACKCHANNEL_PATTERN.match(text) and len(text.split()) <= 4:
        return True
    return False


def normalize_role_from_role_col(role_value):
    if pd.isna(role_value):
        return "unknown"
    role = str(role_value).strip().lower()

    buyer_terms = ["buyer", "customer", "client", "caller", "prospect", "lead"]
    seller_terms = ["seller", "dealer", "agent", "sales", "salesperson", "representative", "rep"]

    if any(term in role for term in buyer_terms):
        return "buyer"
    if any(term in role for term in seller_terms):
        return "seller"
    return "unknown"


def normalize_role_from_speaker_id(speaker_id):
    try:
        speaker_id_int = int(speaker_id)
        return SPEAKER_ID_TO_ROLE.get(speaker_id_int, "unknown")
    except Exception:
        return "unknown"


def standardize_dataset(dataset_name, cfg):
    path = cfg["path"]
    if not path.exists():
        raise FileNotFoundError(f"Missing input file for {dataset_name}: {path}")

    raw = pd.read_csv(path, encoding="utf-8-sig")
    df = raw.copy()

    # Standard identifiers
    df["dataset_name"] = dataset_name
    df["source_file"] = path.name

    df["conversation_id"] = df[cfg["conversation_col"]].astype(str)
    df["speaker_id"] = df[cfg["speaker_col"]]

    # Text
    df["text"] = df[cfg["text_col"]].fillna("").astype(str).str.strip()

    # Turn order
    if "turn_col" in cfg and cfg.get("turn_col") in df.columns:
        df["turn_index"] = df[cfg["turn_col"]]
    else:
        df["turn_index"] = df.groupby("conversation_id").cumcount()

    # Time columns
    if cfg.get("start_col") and cfg["start_col"] in df.columns:
        df["start_time"] = pd.to_numeric(df[cfg["start_col"]], errors="coerce")
    else:
        df["start_time"] = np.nan

    if cfg.get("end_col") and cfg["end_col"] in df.columns:
        df["end_time"] = pd.to_numeric(df[cfg["end_col"]], errors="coerce")
    else:
        df["end_time"] = np.nan

    if cfg.get("duration_col") and cfg["duration_col"] in df.columns:
        df["duration_min"] = pd.to_numeric(df[cfg["duration_col"]], errors="coerce")
    else:
        df["duration_min"] = np.nan

    # Outcome
    outcome_col = cfg.get("outcome_col")
    if outcome_col and outcome_col in df.columns:
        df["outcome"] = df[outcome_col]
    else:
        df["outcome"] = np.nan

    # Role
    if cfg.get("role_col") and cfg["role_col"] in df.columns:
        df["role"] = df[cfg["role_col"]].apply(normalize_role_from_role_col)
    else:
        df["role"] = df["speaker_id"].apply(normalize_role_from_speaker_id)

    df["is_backchannel"] = df["text"].apply(is_backchannel)

    # Stable row id for merging predictions back after combined inference
    df["global_row_id"] = [
        f"{dataset_name}__{i}" for i in range(len(df))
    ]

    ordered_cols = [
        "global_row_id", "dataset_name", "source_file",
        "conversation_id", "turn_index", "speaker_id", "role",
        "start_time", "end_time", "duration_min", "text", "outcome",
        "is_backchannel",
    ]

    return df[ordered_cols], raw


def load_all_datasets_parallel():
    standardized = {}
    raw_inputs = {}

    with ThreadPoolExecutor(max_workers=min(3, len(DATASETS))) as executor:
        futures = {
            executor.submit(standardize_dataset, name, cfg): name
            for name, cfg in DATASETS.items()
        }

        for future in as_completed(futures):
            name = futures[future]
            std_df, raw_df = future.result()
            standardized[name] = std_df
            raw_inputs[name] = raw_df

    # Keep config order
    standardized = {name: standardized[name] for name in DATASETS.keys()}
    raw_inputs = {name: raw_inputs[name] for name in DATASETS.keys()}

    return standardized, raw_inputs


standardized_dfs, raw_inputs = load_all_datasets_parallel()

for name, df in standardized_dfs.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    print("  conversations:", df["conversation_id"].nunique())
    print("  roles:", df["role"].value_counts(dropna=False).to_dict())
    print("  backchannels:", int(df["is_backchannel"].sum()))


dealerships_nego: 2,496 rows x 13 columns
  conversations: 48
  roles: {'buyer': 1240, 'seller': 1238, 'unknown': 18}
  backchannels: 16
emaad_sales: 3,414 rows x 13 columns
  conversations: 200
  roles: {'buyer': 1857, 'seller': 1557}
  backchannels: 0
heddaya_nego: 7,565 rows x 13 columns
  conversations: 178
  roles: {'buyer': 3795, 'seller': 3770}
  backchannels: 1181


In [ ]:
# ============================================================
# 4. LOAD GOEMOTIONS MODEL
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

labels = model.config.id2label
goemotion_labels = [labels[i] for i in range(len(labels))]

print(f"Loaded GoEmotions model with {len(goemotion_labels)} labels:")
print(goemotion_labels)

if EXCLUDE_NEUTRAL_FROM_SST:
    sst_emotion_labels = [label for label in goemotion_labels if label != "neutral"]
else:
    sst_emotion_labels = goemotion_labels.copy()

print(f"Raw GoEmotions labels saved: {len(goemotion_labels)}")
print(f"SST semantic-space labels used: {len(sst_emotion_labels)}")


In [7]:
# ============================================================
# 5. BATCH GOEMOTIONS INFERENCE ACROSS ALL THREE FILES
# ============================================================

def predict_goemotions(texts, batch_size=32, max_length=256):
    all_probs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="GoEmotions inference", disable=DISABLE_PROGRESS_BARS):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            logits = model(**encoded).logits
            probs = torch.sigmoid(logits).detach().cpu().numpy()

        all_probs.append(probs)

    return np.vstack(all_probs)


# Combine substantive turns from all datasets into one inference table.
substantive_all = []
for name, df in standardized_dfs.items():
    substantive_all.append(df.loc[~df["is_backchannel"]].copy())

substantive_all = pd.concat(substantive_all, ignore_index=True)

print("Total substantive turns across files:", len(substantive_all))

texts = substantive_all["text"].fillna("").astype(str).tolist()
goemo_probs = predict_goemotions(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)

goemo_full_df = pd.DataFrame(
    goemo_probs,
    columns=[f"goemo_{label}" for label in goemotion_labels]
)

goemo_full_df["goemo_dominant_emotion"] = goemo_full_df[[f"goemo_{label}" for label in goemotion_labels]].idxmax(axis=1).str.replace("goemo_", "", regex=False)
goemo_full_df["goemo_dominant_score"] = goemo_full_df[[f"goemo_{label}" for label in goemotion_labels]].max(axis=1)
goemo_full_df["goemo_emotion_intensity"] = goemo_full_df[[f"goemo_{label}" for label in goemotion_labels]].sum(axis=1)

substantive_with_goemo = pd.concat(
    [substantive_all.reset_index(drop=True), goemo_full_df.reset_index(drop=True)],
    axis=1
)

print("Combined inference output shape:", substantive_with_goemo.shape)
substantive_with_goemo.head()


Total substantive turns across files: 12278


GoEmotions inference:   0%|          | 0/384 [00:00<?, ?it/s]

Combined inference output shape: (12278, 44)


,global_row_id,dataset_name,source_file,conversation_id,turn_index,speaker_id,role,start_time,end_time,duration_min,...,goemo_pride,goemo_realization,goemo_relief,goemo_remorse,goemo_sadness,goemo_surprise,goemo_neutral,goemo_dominant_emotion,goemo_dominant_score,goemo_emotion_intensity
0,dealerships_nego__0,dealerships_nego,dealerships-nego.csv,2026_1,0,1,seller,0.000000,2.599998,NaN,...,0.000180,0.003453,0.001256,0.007730,0.008767,0.002033,0.383396,neutral,0.383396,1.081336
1,dealerships_nego__1,dealerships_nego,dealerships-nego.csv,2026_1,1,0,buyer,2.599998,11.800003,NaN,...,0.001279,0.005509,0.001607,0.000767,0.002168,0.017690,0.021376,curiosity,0.751382,1.682960
2,dealerships_nego__2,dealerships_nego,dealerships-nego.csv,2026_1,2,1,seller,11.800003,12.840004,NaN,...,0.000859,0.005483,0.002072,0.000274,0.000678,0.000472,0.620828,neutral,0.620828,1.093354
3,dealerships_nego__3,dealerships_nego,dealerships-nego.csv,2026_1,3,0,buyer,12.840004,19.639999,NaN,...,0.000404,0.034998,0.001023,0.000219,0.000860,0.000842,0.698061,neutral,0.698061,1.090367
4,dealerships_nego__4,dealerships_nego,dealerships-nego.csv,2026_1,4,1,seller,19.760002,21.000000,NaN,...,0.000062,0.005695,0.000134,0.000274,0.000759,0.006361,0.621635,neutral,0.621635,1.277390


In [8]:
# ============================================================
# 6. SPLIT-HALF CCA DIMENSIONALITY DISCOVERY
# ============================================================

def safe_corr(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    if len(x) < 3:
        return 0.0
    if np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return 0.0
    r = pearsonr(x, y)[0]
    if np.isnan(r):
        return 0.0
    return float(r)


def split_half_cca_reliability(texts, emotion_matrix, emotion_labels, n_splits=50, random_seed=42):
    '''
    Estimates reliability of each emotion dimension by repeatedly splitting turns,
    learning a shared text-emotion CCA space, and testing whether each held-out
    emotion dimension is recoverable from the text-derived canonical scores.

    This is intentionally lightweight enough for Colab:
    TF-IDF -> TruncatedSVD text features -> CCA with emotion probabilities.
    '''
    n = len(texts)
    rng = np.random.default_rng(random_seed)

    if n < 20:
        return pd.DataFrame({
            "dimension": emotion_labels,
            "reliability_r": [0.0] * len(emotion_labels),
            "retained": [False] * len(emotion_labels)
        })

    # TF-IDF is fit once per dataset for speed and stable vocabulary.
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        min_df=2,
        max_df=0.95,
        ngram_range=(1, 2),
        max_features=10000
    )

    X_tfidf = vectorizer.fit_transform(texts)
    vocab_size = len(vectorizer.vocabulary_)

    # SVD components need to respect matrix size.
    max_components = min(100, X_tfidf.shape[0] - 1, X_tfidf.shape[1] - 1)
    max_components = max(2, max_components)

    svd = TruncatedSVD(n_components=max_components, random_state=random_seed)
    X = svd.fit_transform(X_tfidf)

    Y = np.asarray(emotion_matrix, dtype=float)
    n_emotions = Y.shape[1]

    split_rs = {label: [] for label in emotion_labels}

    for split_id in tqdm(range(n_splits), desc="Split-half CCA", disable=DISABLE_PROGRESS_BARS):
        idx = rng.permutation(n)
        half = n // 2
        idx_a, idx_b = idx[:half], idx[half:]

        # Use as many canonical components as possible but keep it stable.
        n_components = min(n_emotions, X.shape[1], len(idx_a) - 1, len(idx_b) - 1)
        if n_components < 1:
            continue

        cca = CCA(n_components=n_components, max_iter=1000)

        try:
            cca.fit(X[idx_a], Y[idx_a])
            Xb_scores, Yb_scores = cca.transform(X[idx_b], Y[idx_b])

            # Correlate each original emotion dimension with the closest
            # canonical emotion score in the held-out half.
            for j, label in enumerate(emotion_labels):
                corrs = [abs(safe_corr(Y[idx_b, j], Yb_scores[:, k])) for k in range(Yb_scores.shape[1])]
                split_rs[label].append(max(corrs) if corrs else 0.0)

        except Exception:
            # If a split fails numerically, record zero contribution.
            for label in emotion_labels:
                split_rs[label].append(0.0)

    reliability = []
    for label in emotion_labels:
        vals = split_rs[label]
        reliability.append(float(np.mean(vals)) if vals else 0.0)

    out = pd.DataFrame({
        "dimension": emotion_labels,
        "reliability_r": reliability,
    })
    out["retained"] = out["reliability_r"] > RELIABILITY_THRESHOLD
    out["threshold"] = RELIABILITY_THRESHOLD
    out["n_splits"] = n_splits
    out["vocab_size_tfidf"] = vocab_size
    out["svd_components"] = max_components

    return out, vocab_size


In [9]:
# ============================================================
# 7. REPORTING AND OUTPUT WRITERS
# ============================================================

def bar(value, scale=0.03, max_blocks=10):
    try:
        n = int(round(float(value) / scale))
    except Exception:
        n = 0
    n = max(0, min(max_blocks, n))
    return "█" * n


def build_stage1_report(
    dataset_name,
    original_df,
    substantive_df,
    backchannel_df,
    full_goemo_df,
    sst_dimensions_df,
    retained_labels,
    vocab_size,
):
    total_turns = len(original_df)
    substantive_turns = len(substantive_df)
    backchannels = len(backchannel_df)
    n_conversations = original_df["conversation_id"].nunique()

    buyer_turns = int((substantive_df["role"] == "buyer").sum())
    seller_turns = int((substantive_df["role"] == "seller").sum())
    unknown_turns = int((substantive_df["role"] == "unknown").sum())

    role_counts = substantive_df.groupby("conversation_id")["role"].apply(lambda s: set(s.dropna()))
    both_roles = int(role_counts.apply(lambda x: {"buyer", "seller"}.issubset(x)).sum())

    sst_cols = [f"goemo_{label}" for label in sst_emotion_labels]
    means = substantive_df[sst_cols].mean().rename(lambda x: x.replace("goemo_", "")).sort_values(ascending=False)

    role_means = (
        substantive_df
        .loc[substantive_df["role"].isin(["buyer", "seller"])]
        .groupby("role")[sst_cols]
        .mean()
    )

    if {"buyer", "seller"}.issubset(role_means.index):
        diffs = []
        for col in sst_cols:
            label = col.replace("goemo_", "")
            buyer = role_means.loc["buyer", col]
            seller = role_means.loc["seller", col]
            diffs.append((label, buyer, seller, abs(buyer - seller)))
        diffs = sorted(diffs, key=lambda x: x[3], reverse=True)[:5]
    else:
        diffs = []

    lines = []
    lines.append("=" * 60)
    lines.append(f"STAGE 1: SST EMBEDDING PIPELINE — {dataset_name}")
    lines.append("=" * 60)
    lines.append("")
    lines.append("[Step 1] Loading data and filtering backchannels...")
    lines.append(f"  Total turns        : {total_turns:,}")
    lines.append(f"  Substantive turns  : {substantive_turns:,} ({substantive_turns / max(total_turns, 1) * 100:.1f}%)")
    lines.append(f"  Backchannels       : {backchannels:,} ({backchannels / max(total_turns, 1) * 100:.1f}%)")
    lines.append(f"  Conversations      : {n_conversations:,}")
    lines.append("")
    lines.append("[Step 2] Assigning buyer/seller roles...")
    lines.append(f"  Buyer turns  : {buyer_turns:,}")
    lines.append(f"  Seller turns : {seller_turns:,}")
    lines.append(f"  Unknown      : {unknown_turns:,}")
    lines.append(f"  Conversations with both roles: {both_roles}/{n_conversations}")
    lines.append("")
    lines.append(f"[Step 3] Building {len(sst_emotion_labels)}-dim GoEmotions semantic space...")
    lines.append(f"  Embedding matrix shape: {substantive_turns} x {len(sst_emotion_labels)}")
    lines.append(f"  Vocab size (TF-IDF)   : {vocab_size:,}")
    lines.append("")
    lines.append("  Mean activation per emotion dimension:")
    for label, value in means.items():
        lines.append(f"    {label:<18} {value:0.4f}  {bar(value)}")
    lines.append("")
    lines.append("  Buyer vs Seller mean activation (top 5 differences):")
    if diffs:
        for label, buyer, seller, delta in diffs:
            lines.append(f"    {label:<18} buyer={buyer:0.4f}  seller={seller:0.4f}  Δ={delta:0.4f}")
    else:
        lines.append("    Not available: both buyer and seller roles were not identified.")
    lines.append("")
    lines.append(f"[Step 4] Split-half CCA dimensionality discovery (n_splits={N_SPLITS})...")
    lines.append("")
    lines.append(f"  Split-half reliability per dimension (threshold r > {RELIABILITY_THRESHOLD}):")

    for i, row in sst_dimensions_df.reset_index(drop=True).iterrows():
        mark = "✓" if bool(row["retained"]) else "✗"
        lines.append(f"    [{mark}] Dim {i + 1:>2} ({row['dimension']:<18})  r = {row['reliability_r']:0.4f}")

    lines.append("")
    lines.append(f"  Retained dimensions : {len(retained_labels)} / {len(sst_emotion_labels)}")
    lines.append(f"  SST dimensionality  : {len(retained_labels)}")
    lines.append("")
    lines.append(f"[Step 5] Output files written to {OUTPUT_DIR}/{dataset_name}/")
    lines.append(f"  stage1_turns_embedded.csv          — {substantive_turns:,} turns x {len(retained_labels)} retained emotion dims")
    lines.append(f"  stage1_goemotions_features_full28.csv — {substantive_turns:,} turns x {len(goemotion_labels)} GoEmotions dims")
    lines.append(f"  stage1_sst_dimensions.csv          — {len(sst_emotion_labels)} dims + reliability")
    lines.append(f"  stage1_backchannels.csv            — {backchannels:,} backchannel turns")
    lines.append(f"  stage1_report.txt                  — structured Stage 1 report")
    lines.append("")
    lines.append("=" * 60)
    lines.append("STAGE 1 COMPLETE")
    lines.append(f"SST emotional space: {len(retained_labels)}-dimensional")
    lines.append("Ready for Stage 2: coupled SSM on buyer/seller trajectories")
    lines.append("=" * 60)

    return "\n".join(lines)


def write_dataset_outputs(dataset_name, standardized_df, substantive_df):
    dataset_out = OUTPUT_DIR / dataset_name
    dataset_out.mkdir(parents=True, exist_ok=True)

    # Backchannels
    backchannels = standardized_df.loc[standardized_df["is_backchannel"]].copy()

    # Full 28 GoEmotions output
    full_cols = [f"goemo_{label}" for label in goemotion_labels]
    meta_cols = [
        "global_row_id", "dataset_name", "source_file",
        "conversation_id", "turn_index", "speaker_id", "role",
        "start_time", "end_time", "duration_min", "text", "outcome",
    ]
    full_goemo = substantive_df[
        meta_cols + full_cols + [
            "goemo_dominant_emotion",
            "goemo_dominant_score",
            "goemo_emotion_intensity",
        ]
    ].copy()

    # SST dimensionality discovery
    sst_dimensions, retained_labels, vocab_size = discover_sst_dimensions(
        dataset_df=substantive_df,
        dataset_name=dataset_name,
    )

    retained_cols = [f"goemo_{label}" for label in retained_labels]

    embedded = substantive_df[meta_cols + retained_cols].copy()
    embedded = embedded.rename(columns={f"goemo_{label}": label for label in retained_labels})

    # Save CSVs
    full_goemo.to_csv(dataset_out / "stage1_goemotions_features_full28.csv", index=False)
    embedded.to_csv(dataset_out / "stage1_turns_embedded.csv", index=False)
    sst_dimensions.to_csv(dataset_out / "stage1_sst_dimensions.csv", index=False)
    backchannels.to_csv(dataset_out / "stage1_backchannels.csv", index=False)

    # Structured report
    report = build_stage1_report(
        dataset_name=dataset_name,
        original_df=standardized_df,
        substantive_df=substantive_df,
        backchannel_df=backchannels,
        full_goemo_df=full_goemo,
        sst_dimensions_df=sst_dimensions,
        retained_labels=retained_labels,
        vocab_size=vocab_size,
    )

    with open(dataset_out / "stage1_report.txt", "w", encoding="utf-8") as f:
        f.write(report)

    print(report)
    print("\n")

    return {
        "dataset_name": dataset_name,
        "total_turns": len(standardized_df),
        "substantive_turns": len(substantive_df),
        "backchannels": len(backchannels),
        "conversations": standardized_df["conversation_id"].nunique(),
        "retained_sst_dimensions": len(retained_labels),
        "output_dir": str(dataset_out),
    }


In [10]:
# ============================================================
# 8. SPLIT COMBINED INFERENCE BACK TO DATASETS AND WRITE OUTPUTS
# ============================================================

summary_rows = []

for dataset_name in DATASETS.keys():
    standardized_df = standardized_dfs[dataset_name]

    substantive_ids = standardized_df.loc[~standardized_df["is_backchannel"], "global_row_id"]
    substantive_df = substantive_with_goemo.loc[
        substantive_with_goemo["global_row_id"].isin(substantive_ids)
    ].copy()

    # Preserve original turn order
    substantive_df = substantive_df.sort_values(["conversation_id", "turn_index"]).reset_index(drop=True)

    summary = write_dataset_outputs(
        dataset_name=dataset_name,
        standardized_df=standardized_df,
        substantive_df=substantive_df,
    )
    summary_rows.append(summary)

manifest = pd.DataFrame(summary_rows)
manifest.to_csv(OUTPUT_DIR / "stage1_manifest.csv", index=False)

print("Manifest written to:", OUTPUT_DIR / "stage1_manifest.csv")
manifest


Split-half CCA:   0%|          | 0/50 [00:00<?, ?it/s]

STAGE 1: SST EMBEDDING PIPELINE — dealerships_nego

[Step 1] Loading data and filtering backchannels...
  Total turns        : 2,496
  Substantive turns  : 2,480 (99.4%)
  Backchannels       : 16 (0.6%)
  Conversations      : 48

[Step 2] Assigning buyer/seller roles...
  Buyer turns  : 1,233
  Seller turns : 1,229
  Unknown      : 18
  Conversations with both roles: 48/48

[Step 3] Building 27-dim GoEmotions semantic space...
  Embedding matrix shape: 2480 x 27
  Vocab size (TF-IDF)   : 4,990

  Mean activation per emotion dimension:
    approval           0.1762  ██████
    curiosity          0.1658  ██████
    admiration         0.0784  ███
    confusion          0.0579  ██
    disapproval        0.0490  ██
    gratitude          0.0415  █
    optimism           0.0399  █
    desire             0.0227  █
    annoyance          0.0205  █
    excitement         0.0184  █
    caring             0.0174  █
    realization        0.0163  █
    remorse            0.0161  █
    disappointme

Split-half CCA:   0%|          | 0/50 [00:00<?, ?it/s]

STAGE 1: SST EMBEDDING PIPELINE — emaad_sales

[Step 1] Loading data and filtering backchannels...
  Total turns        : 3,414
  Substantive turns  : 3,414 (100.0%)
  Backchannels       : 0 (0.0%)
  Conversations      : 200

[Step 2] Assigning buyer/seller roles...
  Buyer turns  : 1,857
  Seller turns : 1,557
  Unknown      : 0
  Conversations with both roles: 200/200

[Step 3] Building 27-dim GoEmotions semantic space...
  Embedding matrix shape: 3414 x 27
  Vocab size (TF-IDF)   : 4,763

  Mean activation per emotion dimension:
    approval           0.2024  ███████
    curiosity          0.2012  ███████
    gratitude          0.1803  ██████
    admiration         0.1243  ████
    confusion          0.0584  ██
    caring             0.0266  █
    optimism           0.0235  █
    joy                0.0234  █
    disapproval        0.0213  █
    excitement         0.0187  █
    realization        0.0148  
    desire             0.0093  
    love               0.0075  
    annoyance  

Split-half CCA:   0%|          | 0/50 [00:00<?, ?it/s]

STAGE 1: SST EMBEDDING PIPELINE — heddaya_nego

[Step 1] Loading data and filtering backchannels...
  Total turns        : 7,565
  Substantive turns  : 6,384 (84.4%)
  Backchannels       : 1,181 (15.6%)
  Conversations      : 178

[Step 2] Assigning buyer/seller roles...
  Buyer turns  : 3,243
  Seller turns : 3,141
  Unknown      : 0
  Conversations with both roles: 178/178

[Step 3] Building 27-dim GoEmotions semantic space...
  Embedding matrix shape: 6384 x 27
  Vocab size (TF-IDF)   : 8,761

  Mean activation per emotion dimension:
    approval           0.1444  █████
    admiration         0.0663  ██
    curiosity          0.0488  ██
    confusion          0.0385  █
    disapproval        0.0371  █
    optimism           0.0259  █
    realization        0.0200  █
    desire             0.0192  █
    love               0.0178  █
    remorse            0.0150  
    gratitude          0.0140  
    excitement         0.0126  
    disappointment     0.0114  
    joy                0.0

,dataset_name,total_turns,substantive_turns,backchannels,conversations,retained_sst_dimensions,output_dir
0,dealerships_nego,2496,2480,16,48,27,/content/optimal-nego/outputs/dealerships_nego
1,emaad_sales,3414,3414,0,200,27,/content/optimal-nego/outputs/emaad_sales
2,heddaya_nego,7565,6384,1181,178,27,/content/optimal-nego/outputs/heddaya_nego


In [11]:
# ============================================================
# 9. OPTIONAL: ZIP ALL OUTPUTS FOR DOWNLOAD
# ============================================================

import shutil

zip_base = str(BASE_DIR / "stage1_outputs")
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)

print("Created:", zip_path)


Created: /content/optimal-nego/stage1_outputs.zip


## Output folder structure

After running the notebook, your outputs will be organized as:

```text
/content/optimal-nego/outputs/
├── stage1_manifest.csv
├── dealerships_nego/
│   ├── stage1_turns_embedded.csv
│   ├── stage1_goemotions_features_full28.csv
│   ├── stage1_sst_dimensions.csv
│   ├── stage1_backchannels.csv
│   └── stage1_report.txt
├── emaad_sales/
│   └── ...
└── heddaya_nego/
    └── ...
```

The final optional cell also creates:

```text
/content/optimal-nego/stage1_outputs.zip
```
